### gold testing 3 final models

In [ ]:
import torch
#import evaluate
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer, XLMRobertaModel, XLMRobertaTokenizerFast, XLMRobertaForTokenClassification
from transformers import get_linear_schedule_with_warmup, EarlyStoppingCallback, AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
import torch.nn as nn
import pandas as pd
import numpy as np
#from sklearn.metrics import classification_report
from seqeval.metrics import f1_score, classification_report, accuracy_score, precision_score, recall_score
import random
from tqdm import tqdm
import ast
import os

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch
!pip install transformers
!pip install seqeval
!pip install datasets
#!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 25.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
multi_path_plo = '/content/drive/MyDrive/xlm-on-augmented-plomatch'
multi_path_kmeans = '/content/drive/MyDrive/xlm-on-augmented-kmeans'
multi_path_dbscan = '/content/drive/MyDrive/xlm-on-augmented-dbscan'

In [ ]:
# plo match model
multi_tokenizer = AutoTokenizer.from_pretrained(multi_path_plo)
multi_model = AutoModelForTokenClassification.from_pretrained(multi_path_plo)

In [38]:
# kmeans model
multi_tokenizer_k = AutoTokenizer.from_pretrained(multi_path_kmeans)
multi_model_k = AutoModelForTokenClassification.from_pretrained(multi_path_kmeans)

In [51]:
# dbscan model
multi_tokenizer_d = AutoTokenizer.from_pretrained(multi_path_dbscan)
multi_model_d = AutoModelForTokenClassification.from_pretrained(multi_path_dbscan)

In [ ]:
multi_data_collator = DataCollatorForTokenClassification(tokenizer=multi_tokenizer)

In [39]:
multi_data_collator_k = DataCollatorForTokenClassification(tokenizer=multi_tokenizer_k)

In [52]:
multi_data_collator_d = DataCollatorForTokenClassification(tokenizer=multi_tokenizer_d)

In [ ]:
label_list = ['O', 'B-LOC', 'I-LOC', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG']
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}

In [53]:
def multi_tokenize_and_align_labels(example):
    tokenized_inputs = multi_tokenizer_d(  #############################################
        example['translated_text'],
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=128
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label_to_id[example['iob_tags'][word_idx]])
        else:
            labels.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

In [ ]:
def safe_eval(val):
    try:
        return ast.literal_eval(val)
    except:
        return None  # or []

загрузка 3 gold

BE

In [ ]:
eval_be = pd.read_csv('BE_GOLD.csv')
eval_be['translated_text'] = eval_be['translated_text'].apply(safe_eval)
eval_be['iob_tags'] = eval_be['iob_tags'].apply(safe_eval)

In [ ]:
mismatch = eval_be[eval_be['translated_text'].str.len() != eval_be['iob_tags'].str.len()]
print(f'Mismatches found: {len(mismatch)}')

Mismatches found: 0


In [ ]:
eval_be = eval_be[eval_be['translated_text'].str.len() == eval_be['iob_tags'].str.len()]

In [ ]:
be_dataset = Dataset.from_pandas(eval_be[['translated_text', 'iob_tags']])

In [54]:
be_tokenized_dataset = be_dataset.map(multi_tokenize_and_align_labels)

Map:   0%|          | 0/725 [00:00<?, ? examples/s]

SK

In [ ]:
eval_sk = pd.read_csv('WIKIGOLDSK_nomisc.csv')
eval_sk['translated_text'] = eval_sk['translated_text'].apply(safe_eval)
eval_sk['iob_tags'] = eval_sk['iob_tags'].apply(safe_eval)

In [ ]:
mismatch = eval_sk[eval_sk['translated_text'].str.len() != eval_sk['iob_tags'].str.len()]
print(f'Mismatches found: {len(mismatch)}')

In [ ]:
eval_sk = eval_sk[eval_sk['translated_text'].str.len() == eval_sk['iob_tags'].str.len()]

In [ ]:
sk_dataset = Dataset.from_pandas(eval_sk[['translated_text', 'iob_tags']])

In [55]:
sk_tokenized_dataset = sk_dataset.map(multi_tokenize_and_align_labels)

Map:   0%|          | 0/5781 [00:00<?, ? examples/s]

SL

In [24]:
eval_sl = pd.read_csv('SL_GOLD.csv')
eval_sl['translated_text'] = eval_sl['translated_text'].apply(safe_eval)
eval_sl['iob_tags'] = eval_sl['iob_tags'].apply(safe_eval)

In [25]:
mismatch = eval_sl[eval_sl['translated_text'].str.len() != eval_sl['iob_tags'].str.len()]
print(f'Mismatches found: {len(mismatch)}')

Mismatches found: 0


In [26]:
eval_sl = eval_sl[eval_sl['translated_text'].str.len() == eval_sl['iob_tags'].str.len()]

In [27]:
sl_dataset = Dataset.from_pandas(eval_sl[['translated_text', 'iob_tags']])

In [56]:
sl_tokenized_dataset = sl_dataset.map(multi_tokenize_and_align_labels)

Map:   0%|          | 0/685 [00:00<?, ? examples/s]

результаты

In [29]:
def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []

    for pred, label in zip(preds, labels):
        temp_pred = []
        temp_label = []
        for p, l in zip(pred, label):
            if l != -100:
                temp_label.append(id_to_label[l])
                temp_pred.append(id_to_label[p])
        true_labels.append(temp_label)
        true_preds.append(temp_pred)

    report = classification_report(true_labels, true_preds, output_dict=True)

    return {
        'accuracy': accuracy_score(true_labels, true_preds),
        'f1': f1_score(true_labels, true_preds),
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall']
    }


In [30]:
def compute_metrics_noO(p):
    '''по классам и без О'''
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, label in zip(predictions, labels):
        sentence_preds = []
        sentence_labels = []

        for p_id, l_id in zip(pred, label):
            if l_id == -100:
                continue
            sentence_preds.append(id_to_label[p_id])
            sentence_labels.append(id_to_label[l_id])

        true_predictions.append(sentence_preds)
        true_labels.append(sentence_labels)

    report = classification_report(true_labels, true_predictions, output_dict=True, zero_division=0)

    results = {}
    for label, scores in report.items():
        if label == 'O' or label in ['accuracy', 'macro avg', 'weighted avg']:
            continue
        results[f'{label}_f1'] = scores['f1-score']
        results[f'{label}_precision'] = scores['precision']
        results[f'{label}_recall'] = scores['recall']

    # без 'O'
    f1_scores = [v for k, v in results.items() if k.endswith('_f1')]
    results['macro_f1_excl_O'] = np.mean(f1_scores)

    #  overall accuracy
    results['accuracy'] = accuracy_score(true_labels, true_predictions)

    return results

для PLO match

In [31]:
# dummy training args
args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=8,
)

multi_trainer_plo = Trainer(
    model=multi_model,
    args=args,
    tokenizer=multi_tokenizer,
    compute_metrics=compute_metrics,
    data_collator=multi_data_collator
)

<ipython-input-31-30386b03033a>:7: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  multi_trainer_plo = Trainer(


In [32]:
# multi be
multi_metrics_plo = multi_trainer_plo.evaluate(be_tokenized_dataset)
print(multi_metrics_plo)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sol-arina (sol-arina-russian-state-university-for-the-humanities) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'eval_loss': 1.0155448913574219, 'eval_model_preparation_time': 0.0047, 'eval_accuracy': 0.8696541148406883, 'eval_f1': 0.38931297709923657, 'eval_precision': 0.7000522763995818, 'eval_recall': 0.24369201220551107, 'eval_runtime': 6.12, 'eval_samples_per_second': 118.464, 'eval_steps_per_second': 14.869}


In [33]:
# multi sk
multi_metrics1_plo = multi_trainer_plo.evaluate(sk_tokenized_dataset)
print(multi_metrics1_plo)

{'eval_loss': 0.7860398292541504, 'eval_model_preparation_time': 0.0047, 'eval_accuracy': 0.9026813456595957, 'eval_f1': 0.333268178264269, 'eval_precision': 0.7138902881139889, 'eval_recall': 0.22565086946193488, 'eval_runtime': 45.4739, 'eval_samples_per_second': 127.128, 'eval_steps_per_second': 15.899}


In [34]:
# multi sl
multi_metrics2_plo = multi_trainer_plo.evaluate(sl_tokenized_dataset)
print(multi_metrics2_plo)

{'eval_loss': 0.7439053654670715, 'eval_model_preparation_time': 0.0047, 'eval_accuracy': 0.9046482856132909, 'eval_f1': 0.4557124518613607, 'eval_precision': 0.8122316133769832, 'eval_recall': 0.3307555854927838, 'eval_runtime': 5.3788, 'eval_samples_per_second': 127.352, 'eval_steps_per_second': 15.989}


In [35]:
# multi be no o
multi_trainer_plo.compute_metrics = compute_metrics_noO
multi_be_metrics = multi_trainer_plo.evaluate(be_tokenized_dataset)
print(multi_be_metrics)

{'eval_loss': 1.0155448913574219, 'eval_model_preparation_time': 0.0047, 'eval_LOC_f1': 0.2848297213622291, 'eval_LOC_precision': 0.8263473053892215, 'eval_LOC_recall': 0.17206982543640897, 'eval_ORG_f1': 0.1986754966887417, 'eval_ORG_precision': 0.4166666666666667, 'eval_ORG_recall': 0.13043478260869565, 'eval_PER_f1': 0.5714285714285714, 'eval_PER_precision': 0.8571428571428571, 'eval_PER_recall': 0.42857142857142855, 'eval_micro avg_f1': 0.38931297709923657, 'eval_micro avg_precision': 0.8095238095238095, 'eval_micro avg_recall': 0.2562814070351759, 'eval_macro_f1_excl_O': 0.3610616916446947, 'eval_accuracy': 0.8696541148406883, 'eval_runtime': 5.4252, 'eval_samples_per_second': 133.635, 'eval_steps_per_second': 16.773}


In [36]:
# multi sk no o
multi_trainer_plo.compute_metrics = compute_metrics_noO
multi_sk_metrics = multi_trainer_plo.evaluate(sk_tokenized_dataset)
print(multi_sk_metrics)

{'eval_loss': 0.7860398292541504, 'eval_model_preparation_time': 0.0047, 'eval_LOC_f1': 0.23519903010709234, 'eval_LOC_precision': 0.6736111111111112, 'eval_LOC_recall': 0.14247246022031823, 'eval_ORG_f1': 0.2133468149646107, 'eval_ORG_precision': 0.6011396011396012, 'eval_ORG_recall': 0.12968653964351567, 'eval_PER_f1': 0.551891074130106, 'eval_PER_precision': 0.8669201520912547, 'eval_PER_recall': 0.4047936085219707, 'eval_micro avg_f1': 0.333268178264269, 'eval_micro avg_precision': 0.7520952801058668, 'eval_micro avg_recall': 0.21406151914626492, 'eval_macro_f1_excl_O': 0.3334262743665195, 'eval_accuracy': 0.9026813456595957, 'eval_runtime': 52.3022, 'eval_samples_per_second': 110.531, 'eval_steps_per_second': 13.824}


In [37]:
#multi sl no o
multi_trainer_plo.compute_metrics = compute_metrics_noO
multi_sl_metrics = multi_trainer_plo.evaluate(sl_tokenized_dataset)
print(multi_sl_metrics)

{'eval_loss': 0.7439053654670715, 'eval_model_preparation_time': 0.0047, 'eval_LOC_f1': 0.3170441001191895, 'eval_LOC_precision': 0.8159509202453987, 'eval_LOC_recall': 0.19674556213017752, 'eval_ORG_f1': 0.37398373983739835, 'eval_ORG_precision': 0.7666666666666667, 'eval_ORG_recall': 0.24731182795698925, 'eval_PER_f1': 0.6677852348993288, 'eval_PER_precision': 0.8540772532188842, 'eval_PER_recall': 0.5482093663911846, 'eval_micro avg_f1': 0.4557124518613607, 'eval_micro avg_precision': 0.8333333333333334, 'eval_micro avg_recall': 0.3136042402826855, 'eval_macro_f1_excl_O': 0.4536313816793193, 'eval_accuracy': 0.9046482856132909, 'eval_runtime': 5.5408, 'eval_samples_per_second': 123.629, 'eval_steps_per_second': 15.521}


для K-MEans

In [44]:
args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=8,
)

multi_trainer_kmeans = Trainer(
    model=multi_model_k,
    args=args,
    tokenizer=multi_tokenizer_k,
    compute_metrics=compute_metrics,
    data_collator=multi_data_collator_k
)

<ipython-input-44-3345865e60d1>:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  multi_trainer_kmeans = Trainer(


In [45]:
# multi be
multi_metrics_k = multi_trainer_kmeans.evaluate(be_tokenized_dataset)
print(multi_metrics_k)

{'eval_loss': 1.9263367652893066, 'eval_model_preparation_time': 0.0029, 'eval_accuracy': 0.8177713409439428, 'eval_f1': 0.011412268188302425, 'eval_precision': 0.9166666666666666, 'eval_recall': 0.014110670832357662, 'eval_runtime': 5.7109, 'eval_samples_per_second': 126.951, 'eval_steps_per_second': 15.935}


In [46]:
# multi sk
multi_metrics1_k = multi_trainer_kmeans.evaluate(sk_tokenized_dataset)
print(multi_metrics1_k)

{'eval_loss': 0.9770414233207703, 'eval_model_preparation_time': 0.0029, 'eval_accuracy': 0.8900609919828149, 'eval_f1': 0.20965164589325663, 'eval_precision': 0.6500430502742515, 'eval_recall': 0.1298134229953696, 'eval_runtime': 46.6067, 'eval_samples_per_second': 124.038, 'eval_steps_per_second': 15.513}


In [47]:
# multi sl
multi_metrics2_k = multi_trainer_kmeans.evaluate(sl_tokenized_dataset)
print(multi_metrics2_k)

{'eval_loss': 0.9725775718688965, 'eval_model_preparation_time': 0.0029, 'eval_accuracy': 0.8761046306115235, 'eval_f1': 0.2565597667638484, 'eval_precision': 0.8208377581120944, 'eval_recall': 0.1316583822223344, 'eval_runtime': 7.129, 'eval_samples_per_second': 96.087, 'eval_steps_per_second': 12.063}


In [48]:
# multi be no o
multi_trainer_kmeans.compute_metrics = compute_metrics_noO
multi_be_metrics_k = multi_trainer_kmeans.evaluate(be_tokenized_dataset)
print(multi_be_metrics_k)

{'eval_loss': 1.9263367652893066, 'eval_model_preparation_time': 0.0029, 'eval_LOC_f1': 0.0024906600249066002, 'eval_LOC_precision': 1.0, 'eval_LOC_recall': 0.0012468827930174563, 'eval_ORG_f1': 0.06722689075630252, 'eval_ORG_precision': 1.0, 'eval_ORG_recall': 0.034782608695652174, 'eval_PER_f1': 0.012500000000000002, 'eval_PER_precision': 0.75, 'eval_PER_recall': 0.0063025210084033615, 'eval_micro avg_f1': 0.011412268188302425, 'eval_micro avg_precision': 0.8888888888888888, 'eval_micro avg_recall': 0.00574300071787509, 'eval_macro_f1_excl_O': 0.023407454742377883, 'eval_accuracy': 0.8177713409439428, 'eval_runtime': 7.3269, 'eval_samples_per_second': 98.95, 'eval_steps_per_second': 12.42}


In [49]:
# multi sk no o
multi_trainer_kmeans.compute_metrics = compute_metrics_noO
multi_sk_metrics_k = multi_trainer_kmeans.evaluate(sk_tokenized_dataset)
print(multi_sk_metrics_k)

{'eval_loss': 0.9770414233207703, 'eval_model_preparation_time': 0.0029, 'eval_LOC_f1': 0.12219780219780219, 'eval_LOC_precision': 0.5978494623655914, 'eval_LOC_recall': 0.06805385556915544, 'eval_ORG_f1': 0.05513196480938416, 'eval_ORG_precision': 0.6025641025641025, 'eval_ORG_recall': 0.028887523048555623, 'eval_PER_f1': 0.4208173690932312, 'eval_PER_precision': 0.7497155858930603, 'eval_PER_recall': 0.2924988903683977, 'eval_micro avg_f1': 0.20965164589325663, 'eval_micro avg_precision': 0.6919831223628692, 'eval_micro avg_recall': 0.12354048964218456, 'eval_macro_f1_excl_O': 0.20194969549841857, 'eval_accuracy': 0.8900609919828149, 'eval_runtime': 51.6888, 'eval_samples_per_second': 111.842, 'eval_steps_per_second': 13.988}


In [50]:
#multi sl no o
multi_trainer_kmeans.compute_metrics = compute_metrics_noO
multi_sl_metrics_k = multi_trainer_kmeans.evaluate(sl_tokenized_dataset)
print(multi_sl_metrics_k)

{'eval_loss': 0.9725775718688965, 'eval_model_preparation_time': 0.0029, 'eval_LOC_f1': 0.21039290240811154, 'eval_LOC_precision': 0.7345132743362832, 'eval_LOC_recall': 0.1227810650887574, 'eval_ORG_f1': 0.04210526315789474, 'eval_ORG_precision': 1.0, 'eval_ORG_recall': 0.021505376344086023, 'eval_PER_f1': 0.3729508196721311, 'eval_PER_precision': 0.728, 'eval_PER_recall': 0.25068870523415976, 'eval_micro avg_f1': 0.2565597667638484, 'eval_micro avg_precision': 0.7333333333333333, 'eval_micro avg_recall': 0.15547703180212014, 'eval_macro_f1_excl_O': 0.22050218800049645, 'eval_accuracy': 0.8761046306115235, 'eval_runtime': 7.644, 'eval_samples_per_second': 89.612, 'eval_steps_per_second': 11.251}


Для DBSCAN

In [57]:
args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=8,
)

multi_trainer_dbscan = Trainer(
    model=multi_model_d,
    args=args,
    tokenizer=multi_tokenizer_d,
    compute_metrics=compute_metrics,
    data_collator=multi_data_collator_d
)

<ipython-input-57-8e99da29782a>:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  multi_trainer_dbscan = Trainer(


In [58]:
# multi be
multi_metrics_d = multi_trainer_dbscan.evaluate(be_tokenized_dataset)
print(multi_metrics_d)

{'eval_loss': 1.9554299116134644, 'eval_model_preparation_time': 0.0029, 'eval_accuracy': 0.8182825012779008, 'eval_f1': 0.014224751066856329, 'eval_precision': 0.875, 'eval_recall': 0.0155112310564473, 'eval_runtime': 6.0041, 'eval_samples_per_second': 120.75, 'eval_steps_per_second': 15.156}


In [59]:
# multi sk
multi_metrics1_d = multi_trainer_dbscan.evaluate(sk_tokenized_dataset)
print(multi_metrics1_d)

{'eval_loss': 0.9947119355201721, 'eval_model_preparation_time': 0.0029, 'eval_accuracy': 0.891758410372473, 'eval_f1': 0.23007534755385756, 'eval_precision': 0.7061207436502973, 'eval_recall': 0.14685687173400358, 'eval_runtime': 45.947, 'eval_samples_per_second': 125.819, 'eval_steps_per_second': 15.736}


In [60]:
# multi sl
multi_metrics2_d = multi_trainer_dbscan.evaluate(sl_tokenized_dataset)
print(multi_metrics2_d)

{'eval_loss': 0.9770668148994446, 'eval_model_preparation_time': 0.0029, 'eval_accuracy': 0.8804347826086957, 'eval_f1': 0.3025936599423631, 'eval_precision': 0.8407141888685513, 'eval_recall': 0.17578428789798065, 'eval_runtime': 5.4568, 'eval_samples_per_second': 125.532, 'eval_steps_per_second': 15.76}


In [61]:
# multi be no o
multi_trainer_dbscan.compute_metrics = compute_metrics_noO
multi_be_metrics_d = multi_trainer_dbscan.evaluate(be_tokenized_dataset)
print(multi_be_metrics_d)

{'eval_loss': 1.9554299116134644, 'eval_model_preparation_time': 0.0029, 'eval_LOC_f1': 0.0024906600249066002, 'eval_LOC_precision': 1.0, 'eval_LOC_recall': 0.0012468827930174563, 'eval_ORG_f1': 0.06722689075630252, 'eval_ORG_precision': 1.0, 'eval_ORG_recall': 0.034782608695652174, 'eval_PER_f1': 0.020661157024793392, 'eval_PER_precision': 0.625, 'eval_PER_recall': 0.01050420168067227, 'eval_micro avg_f1': 0.014224751066856329, 'eval_micro avg_precision': 0.7692307692307693, 'eval_micro avg_recall': 0.007178750897343862, 'eval_macro_f1_excl_O': 0.02615086471821471, 'eval_accuracy': 0.8182825012779008, 'eval_runtime': 5.6244, 'eval_samples_per_second': 128.903, 'eval_steps_per_second': 16.18}


In [62]:
# multi sk no o
multi_trainer_dbscan.compute_metrics = compute_metrics_noO
multi_sk_metrics_d = multi_trainer_dbscan.evaluate(sk_tokenized_dataset)
print(multi_sk_metrics_d)

{'eval_loss': 0.9947119355201721, 'eval_model_preparation_time': 0.0029, 'eval_LOC_f1': 0.1226857015391479, 'eval_LOC_precision': 0.6909547738693468, 'eval_LOC_recall': 0.06731946144430845, 'eval_ORG_f1': 0.09458689458689458, 'eval_ORG_precision': 0.6484375, 'eval_ORG_recall': 0.051014136447449294, 'eval_PER_f1': 0.4558869701726844, 'eval_PER_precision': 0.778969957081545, 'eval_PER_recall': 0.322237017310253, 'eval_micro avg_f1': 0.23007534755385756, 'eval_micro avg_precision': 0.7434842249657064, 'eval_micro avg_recall': 0.13609541745134965, 'eval_macro_f1_excl_O': 0.2258087284631461, 'eval_accuracy': 0.891758410372473, 'eval_runtime': 45.1888, 'eval_samples_per_second': 127.93, 'eval_steps_per_second': 16.0}


In [63]:
#multi sl no o
multi_trainer_dbscan.compute_metrics = compute_metrics_noO
multi_sl_metrics_d = multi_trainer_dbscan.evaluate(sl_tokenized_dataset)
print(multi_sl_metrics_d)

{'eval_loss': 0.9770668148994446, 'eval_model_preparation_time': 0.0029, 'eval_LOC_f1': 0.21677419354838712, 'eval_LOC_precision': 0.8484848484848485, 'eval_LOC_recall': 0.1242603550295858, 'eval_ORG_f1': 0.1386138613861386, 'eval_ORG_precision': 0.875, 'eval_ORG_recall': 0.07526881720430108, 'eval_PER_f1': 0.46484374999999994, 'eval_PER_precision': 0.7986577181208053, 'eval_PER_recall': 0.3278236914600551, 'eval_micro avg_f1': 0.3025936599423631, 'eval_micro avg_precision': 0.8203125, 'eval_micro avg_recall': 0.1855123674911661, 'eval_macro_f1_excl_O': 0.2807063662192222, 'eval_accuracy': 0.8804347826086957, 'eval_runtime': 5.5345, 'eval_samples_per_second': 123.769, 'eval_steps_per_second': 15.539}
